In [1]:
import sys
sys.path.append("../")

In [2]:
from tools import download_documentation_data

In [3]:
data = download_documentation_data()

In [4]:
interesting_pages = []

for page in data:
    if len(page['content']) <= 1000:
        continue
    if page['filename'] == 'changelog/changelog.mdx':
        continue

    interesting_pages.append(page)

print(len(data), len(interesting_pages))

95 71


In [82]:
doc = interesting_pages[15]
content = doc['content']

In [84]:
doc['filename']

'docs/platform/datasets_overview.mdx'

In [6]:
print(content)

<Check>
  Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
</Check>

## What is a Dataset?

**Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthetic datasets.

![](/images/dataset_llm.png)

## How to create a Dataset?

You can add Datasets to the platform in multiple ways:

- **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
- **Upload with Reports**. Attach datasets to Reports when running local evaluations. This is optional — you can also upload only summary metrics.
- **Generate synthetic data**. Use built-in platform features to generate synthetic evaluation datasets. (Cloud only).
- **Create from Traces**. During tracing, Evidently automatically generates tabular datasets that can be used for evaluations. (Cloud only).

<Tip>
  **Where do I find the data?** To view all datasets 

In [65]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field
from typing import Literal


In [ ]:
class GeneratedQuestion(BaseModel):
    user_question: str = Field(description="the question we generate")
    reference_answer: str = Field(description="what is the correct answer based on the document we analyzed")
    line_number_start: int = Field(description="the corresponding lines from the reference document where the reference answer starts")
    line_number_end: int = Field(description="the corresponding lines from the reference document where the reference answer ends, chunked from the part of the document that was used for generating the question")
    question_type: Literal["frustrated_beginner", "hurried_developer", "context_assumer", "direct_inquirer"] = Field(description="the persona style used to phrase the question")

class QuestionsResponse(BaseModel):
    questions: list[GeneratedQuestion]


class GeneratedQuestions(BaseModel):
    questions: list[GeneratedQuestion]
    filename: str = Field(description="the source that we used for generating the question")

In [67]:
instructions = """
You are an expert at generating realistic user questions based on technical documentation.
Your task is to analyze the provided documentation page and generate questions that a user might realistically ask.

For each generated question, you must also extract the reference answer directly based on the document's content.
You must also pinpoint the exact starting and ending line numbers in the provided document where this answer is located.

Follow these guidelines:
1. Generate diverse questions covering different aspects of the document.
2. The questions should be phrased naturally, how an actual user would ask them.
3. The reference answer should be comprehensive and directly supported by the text.
4. The line numbers must accurately reflect where the reference answer can be found in the provided document with line numbers attached.
""".strip()

In [68]:
instructions = """
You are an expert at generating realistic user questions based on technical documentation.
Your task is to analyze the provided documentation page and generate questions that a user might realistically ask.

We want to generate questions that represent REAL users who are stuck, frustrated, or searching for specific information — rather than perfectly phrased textbook questions.

When generating questions, randomly adopt one of the following personas/styles for each question:
1. The Frustrated Beginner: Uses non-technical terms, explains what they are trying to achieve (their goal) but are stuck. (e.g. "I can't figure out how to add my csv file, where is the button?")
2. The Hurried Developer: Uses fragments, keywords, or short phrases instead of full sentences. (e.g. "python api upload dataset")
3. The Context-Assumer: Asks a question assuming the bot knows what page they are looking at or what step they are on. (e.g. "It says I can bring existing data, how?")
4. The Direct Inquirer: Asks a straightforward, somewhat technical question in a conversational tone. (e.g. "Can I create synthetic datasets directly from a PDF?")

For each generated question, you must also extract the reference answer directly based on the document's content.
You must also pinpoint the exact starting and ending line numbers in the provided document where this answer is located.

Follow these strict guidelines:
1. DO NOT end questions with phrases like "based on this text" or "according to the documentation."
2. DO NOT make the questions too formal or academic.
3. The reference answer should be comprehensive and directly supported by the text.
4. The line numbers must accurately reflect exactly where the reference answer can be found in the provided document with line numbers attached.
""".strip()

In [69]:
data_generator = Agent(
    name='data_generator',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    output_type=QuestionsResponse
)

In [70]:
def add_line_numbers(document: str) -> str:
    lines = document.splitlines()
    numbered_lines = [f"{i+1}: {line}" for i, line in enumerate(lines)]
    return "\n".join(numbered_lines)

prompt_template = """
Number of questions:
{number_of_questions}

Document:
{numbered_content}
""".strip()


def create_user_prompt(document_content: str) -> str:
    number_of_questions = len(document_content) // 1000
    numbered_content = add_line_numbers(document_content)

    prompt = prompt_template.format(
        number_of_questions=number_of_questions,
        numbered_content=numbered_content
    )

    return prompt.strip()

In [71]:
print(add_line_numbers(content)[:500])

1: <Check>
2:   Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
3: </Check>
4: 
5: ## What is a Dataset?
6: 
7: **Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthetic datasets.
8: 
9: ![](/images/dataset_llm.png)
10: 
11: ## How to create a Dataset?
12: 
13: You can add Datasets to the platform in multiple ways:
14: 
15: - **Upload directly**. Us


In [72]:
user_prompt = create_user_prompt(content)

In [73]:
questions_result = await data_generator.run(user_prompt)

In [74]:
questions = questions_result.output

In [75]:
len(questions.questions)

2

In [76]:
q = questions.questions[1]

In [78]:
print(q.user_question)
print(q.reference_answer)
print(q.line_number_start)
print(q.line_number_end)
print(q.question_type)

how to upload CSV files directly?
You can add Datasets to the platform in multiple ways: - **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
13
15
hurried_developer


In [79]:
numbered_content = add_line_numbers(content)

In [80]:
print(numbered_content[:1000])

1: <Check>
2:   Datasets are available in **Evidently OSS, Cloud** and **Evidently Enterprise**.
3: </Check>
4: 
5: ## What is a Dataset?
6: 
7: **Datasets** are collections of data from your application used for analysis and automated checks. You can bring in existing datasets, capture live data, or create synthetic datasets.
8: 
9: ![](/images/dataset_llm.png)
10: 
11: ## How to create a Dataset?
12: 
13: You can add Datasets to the platform in multiple ways:
14: 
15: - **Upload directly**. Use the UI (Evidently Cloud) to upload CSV files or push datasets via the Python API.
16: - **Upload with Reports**. Attach datasets to Reports when running local evaluations. This is optional — you can also upload only summary metrics.
17: - **Generate synthetic data**. Use built-in platform features to generate synthetic evaluation datasets. (Cloud only).
18: - **Create from Traces**. During tracing, Evidently automatically generates tabular datasets that can be used for evaluations. (Cloud only

In [86]:
generated_question = GeneratedQuestions(
    questions=questions.questions,
    filename=doc['filename']
)

In [89]:
from tqdm.auto import tqdm

In [90]:
results = []

for doc in tqdm(interesting_pages):
    content = doc['content']

    user_prompt = create_user_prompt(content)
    result = await data_generator.run(user_prompt)

    generated_question = GeneratedQuestions(
        questions=result.output.questions,
        filename=doc['filename']
    )

    results.append(generated_question)

  0%|          | 0/71 [00:00<?, ?it/s]

In [92]:
results[0].questions[0]

GeneratedQuestion(user_question="I can't figure out how to create a Dataset object, is there an example?", reference_answer='To create a Dataset object, use `Dataset.from_pandas` with `data_definition`: \n```python\n eval_data = Dataset.from_pandas(\n     source_df,\n     data_definition=DataDefinition()\n )\n```', line_number_start=23, line_number_end=29, question_type='frustrated_beginner')

In [94]:
rows = []

for gqs in results:
    for q in gqs.questions:
        row = {
            'question': q.user_question,
            'reference_answer': q.reference_answer,
            'line_number_start': q.line_number_start,
            'line_number_end': q.line_number_end,
            'question_type': q.question_type,
            'filename': gqs.filename
        }
        rows.append(row)

In [95]:
len(rows)

199

In [99]:
rows[120]

{'question': "How can I classify a text as 'toxic' with HuggingFace?",
 'reference_answer': 'You can call built-in evaluators for some models. For instance, you can add a descriptor like this: eval_df.add_descriptors(descriptors=[HuggingFaceToxicity("question", toxic_label="hate", alias="Toxicity")]).',
 'line_number_start': 63,
 'line_number_end': 68,
 'question_type': 'frustrated_beginner',
 'filename': 'metrics/customize_hf_descriptor.mdx'}

In [98]:
import pandas as pd 

In [101]:
df = pd.DataFrame(rows)

In [102]:
df.to_csv('questions_generated.csv', index=False)